In [ ]:
#google drive의 COSE362-term-project/dataset 폴더와 연결
from google.colab import drive

drive.mount('/content/drive')

!ls /content/drive/MyDrive/COSE362-term-project/dataset

import sys

sys.path.append('/content/drive/MyDrive/COSE362-term-project/dataset')

import os

os.chdir("/content/drive/MyDrive/COSE362-term-project/dataset")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
baseline  crop	landuse  Shapefiles


In [ ]:
!ls
!ls crop/GROW-Africa-Database/

baseline  crop	landuse  Shapefiles
GROW-Africa_LSMS_cropcut.xlsx  GROW-Africa_Regional.csv
GROW-Africa_LSMS_survey.xlsx   GROW-Africa_Regional.xlsx
GROW-Africa_Point.xlsx


In [ ]:
!pip install rasterstats

# **CROP DATASET Preprocessing**

## **1. 지역단위 level 1 선별**

## **2. 작물 그룹화 및 그룹 통계**
### 작물 그룹 단위
  - 1) 주요 곡물
    - 식량 안보에 중요한 탄수화물 공급원
    - 영양상태에 가장 큰 영향을 줄 것이라고 판단 가능
  - 2) 근경 작물
    - 주요 곡물들 다음으로 중요한 작물
  - 3) 콩류 및 두류
    - 단백질 공급원
    - 영양 균형을 판단할 수 있는 지표가 될 것
  - 4) 환금 작물
    - 직접 먹기보다 팔아서 돈을 벌 수 있는 작물
    - 해당 지역의 경제상황 파악 가능

In [ ]:
from rasterstats import zonal_stats
import geopandas as gpd
import glob
import numpy as np
import pandas as pd

# 1. 지역구 지도(Shapefile)를 불러옵니다 (GROW-Africa)
regions = gpd.read_file("Shapefiles/GADM_level1_ECG.shp")

# 2. crop data를 가져옵니다.
growAfricaDB = pd.read_csv("crop/GROW-Africa-Database/GROW-Africa_Regional.csv")

growAfrica_l1 = growAfricaDB[growAfricaDB['level']==1.0]

l1_list = set(growAfrica_l1['admin_1'])
crop_list = set(growAfrica_l1['crop'])

# 작물 분류
# 1. 주요 곡물 : 식량 안보 및 탄수화물 공급원
cereals = [
    'Maize', 'Sorghum', 'Millet', 'Rice', 'Wheat',
    'Teff', 'Barley', 'Fonio', 'Oats', 'Triticale'
]

# 2. 근경작물 : 1 다음으로 중요한 것
roots_tubers = [
    'Cassava', 'Yam', 'Potato', 'Sweet potato', 'Taro'
]

# 3. 콩류 및 두류 : 단백질 공급원
legumes_pulses = [
    'Beans', 'Cowpea', 'Soybean', 'Chickpeas', 'Lentils',
    'Groundnut', 'Pigeon pea', 'Pulses', 'Legumes'
]

# 4. 환금 작물 : 가계 소득원 (식용 외 목적)
cash_crops = [
    'Coffee', 'Cocoa', 'Tobacco', 'Cotton',
    'Sugar cane', 'Sugar beet'
]

crop_group = [cereals, roots_tubers, legumes_pulses, cash_crops]


/tmp/ipython-input-19289812.py:11: DtypeWarning: Columns (22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  growAfricaDB = pd.read_csv("crop/GROW-Africa-Database/GROW-Africa_Regional.csv")


In [ ]:
# 이전 셀 제대로 작동했는지 테스트
print(growAfrica_l1.columns)

growAfrica_l1.head()

crop_group[0]

Index(['GDAM_ID', 'country', 'country_code', 'admin_1', 'admin_2', 'level',
       'crop', 'season_name', 'planting_year', 'planting_month',
       'harvest_year', 'harvest_month', 'crop_production_system', 'areaHa',
       'productionTon', 'yieldTonHa', 'QC_flag_A', 'QC_flag_P', 'QC_flag_Y',
       'Source', 'L0_GID', 'L1_GID', 'L2_GID', 'L3_GID', 'originalname'],
      dtype='object')


['Maize',
 'Sorghum',
 'Millet',
 'Rice',
 'Wheat',
 'Teff',
 'Barley',
 'Fonio',
 'Oats',
 'Triticale']

In [ ]:
# growAfrica의 각 컬럼 요소들 파악하기

len(set(growAfrica_l1['crop']))

34

In [ ]:
df_pivot = growAfrica_l1.pivot_table(
    index=['country', 'country_code','admin_1', 'L1_GID', 'harvest_year'],
    columns='crop',
    values='yieldTonHa'
)

df_pivot

df_pivot['Cereals_Prod_Ton'] = df_pivot[cereals].sum(axis=1)
df_pivot['Roots_tubers_Prod_Ton'] = df_pivot[roots_tubers].sum(axis=1)
df_pivot['Legumes_pulses_Prod_Ton'] = df_pivot[legumes_pulses].sum(axis=1)
df_pivot['Cash_crops_Prod_Ton'] = df_pivot[cash_crops].sum(axis=1)

df_final = df_pivot[['Cereals_Prod_Ton', 'Roots_tubers_Prod_Ton', 'Legumes_pulses_Prod_Ton', 'Cash_crops_Prod_Ton']]

df_final
# 이 데이터에 존재하는 {admin1지역(시 단위), year} 조합 별 단위지역(1ha) 당 'cereal 그룹 작물 생산량', 'roots_tubers 그룹 작물 생산량', 'legumes_pulses 그룹 작물 생산량', 'cash_crop 그룹 작물 생산량'을 정리함
# Disease data 기준으로 합치기 위해 필요한 {admin1지역(시 단위), year} 조합 골라내는 작업은 아직 못함


crop                                                             Cereals_Prod_Ton  \
country  country_code admin_1            L1_GID    harvest_year                     
Algeria  DZA          Adrar              L1DZA0774 1996                  4.854724   
                                                   1998                  9.552425   
                                                   1999                  0.000000   
                      Alger              L1DZA0777 1996                  2.154930   
                                                   1998                  3.960932   
...                                                                           ...   
Zimbabwe ZWE          Matabeleland South L1ZWE3661 2012                  0.069491   
                                                   2015                  0.277564   
                      Midlands           L1ZWE3662 1994                 14.144515   
                                                   2012                  2.232392   
                                                   2015                 13.738672   

crop                                                             Roots_tubers_Prod_Ton  \
country  country_code admin_1            L1_GID    harvest_year                          
Algeria  DZA          Adrar              L1DZA0774 1996                      11.882353   
                                                   1998                       8.835366   
                                                   1999                      10.380952   
                      Alger              L1DZA0777 1996                      14.987013   
                                                   1998                      15.728764   
...                                                                                ...   
Zimbabwe ZWE          Matabeleland South L1ZWE3661 2012                       4.776762   
                                                   2015                      11.173913   
                      Midlands           L1ZWE3662 1994                       0.000000   
                                                   2012                      18.016093   
                                                   2015                       6.069412   

crop                                                             Legumes_pulses_Prod_Ton  \
country  country_code admin_1            L1_GID    harvest_year                            
Algeria  DZA          Adrar              L1DZA0774 1996                         0.000000   
                                                   1998                         1.986667   
                                                   1999                         0.000000   
                      Alger              L1DZA0777 1996                         0.000000   
                                                   1998                         0.000000   
...                                                                                  ...   
Zimbabwe ZWE          Matabeleland South L1ZWE3661 2012                         0.260766   
                                                   2015                         0.455729   
                      Midlands           L1ZWE3662 1994                         0.917087   
                                                   2012                         2.500376   
                                                   2015                         1.334960   

crop                                                             Cash_crops_Prod_Ton  
country  country_code admin_1            L1_GID    harvest_year                       
Algeria  DZA          Adrar              L1DZA0774 1996                     0.000000  
                                                   1998                     0.000000  
                                                   1999                     0.000000  
                      Alger              L1DZA0777 1996                     0.000000  
    

In [ ]:

# CSV 파일로 저장
df_final.to_csv("crop_groupped.csv", index=True)

print("최종 피처 테이블 저장 완료!")

최종 피처 테이블 저장 완료!


In [ ]:
df_pivot

crop                                                             Banana  \
country  country_code admin_1            L1_GID    harvest_year           
Algeria  DZA          Adrar              L1DZA0774 1996             NaN   
                                                   1998             NaN   
                                                   1999             NaN   
                      Alger              L1DZA0777 1996             NaN   
                                                   1998             NaN   
...                                                                 ...   
Zimbabwe ZWE          Matabeleland South L1ZWE3661 2012             NaN   
                                                   2015             NaN   
                      Midlands           L1ZWE3662 1994             NaN   
                                                   2012             NaN   
                                                   2015             NaN   

crop                                                               Barley  \
country  country_code admin_1            L1_GID    harvest_year             
Algeria  DZA          Adrar              L1DZA0774 1996          2.223684   
                                                   1998          3.118280   
                                                   1999               NaN   
                      Alger              L1DZA0777 1996          1.000000   
                                                   1998          2.092511   
...                                                                   ...   
Zimbabwe ZWE          Matabeleland South L1ZWE3661 2012               NaN   
                                                   2015               NaN   
                      Midlands           L1ZWE3662 1994               NaN   
                                                   2012          1.000000   
                                                   2015          8.198198   

crop                                                                Beans  \
country  country_code admin_1            L1_GID    harvest_year             
Algeria  DZA          Adrar              L1DZA0774 1996               NaN   
                                                   1998               NaN   
                                                   1999               NaN   
                      Alger              L1DZA0777 1996               NaN   
                                                   1998               NaN   
...                                                                   ...   
Zimbabwe ZWE          Matabeleland South L1ZWE3661 2012          0.097508   
                                                   2015          0.203876   
                      Midlands           L1ZWE3662 1994               NaN   
                                                   2012          0.514766   
                                                   2015          0.223352   

crop                                                             Cassava  \
country  country_code admin_1            L1_GID    harvest_year            
Algeria  DZA          Adrar              L1DZA0774 1996              NaN   
                                                   1998              NaN   
                                                   1999              NaN   
                      Alger              L1DZA0777 1996              NaN   
                                                   1998              NaN   
...                                                                  ...   
Zimbabwe ZWE          Matabeleland South L1ZWE3661 2012              NaN   
                                                   2015              NaN   
                      Midlands           L1ZWE3662 1994              NaN   
                                                   2012              NaN   
                                                   2015              NaN   

crop                           

# **disease data {region, year} pair에 맞는 데이터 선별**

In [ ]:
...